# Fine-Tuning Foundations and Data

Fine-tuning is the process of adapting a pretrained model to a narrower behavior, domain, style, or task. The model already knows language patterns from pretraining; fine-tuning changes how it uses that knowledge.

This notebook builds the mental model: what types of fine-tuning exist, what data they need, what loss is optimized, and how to inspect a dataset before training.

## What You Will Learn

By the end, you should be able to explain:

- continued pretraining vs supervised fine-tuning,
- instruction tuning vs task-specific fine-tuning,
- full fine-tuning vs parameter-efficient fine-tuning,
- what an instruction dataset looks like,
- what model inputs and labels look like for causal language modeling,
- why data quality usually matters more than clever training flags,
- and how to run basic dataset checks before training.

## 1. The Fine-Tuning Map

| Type | What data looks like | What changes | Use when |
|---|---|---|---|
| Continued pretraining | Raw domain text | Model language/domain distribution | You need domain vocabulary or style adaptation |
| Supervised fine-tuning | Prompt-response pairs | Model behavior on examples | You want the model to imitate desired answers |
| Instruction tuning | Many diverse instruction-response pairs | General instruction following | You want broad helpfulness across tasks |
| Task fine-tuning | Narrow task examples | Task-specific behavior | You need performance on one task |
| Full fine-tuning | Any of the above | All weights | You have enough compute/data and need maximum flexibility |
| LoRA / adapters | Any of the above | Small trainable modules | You want cheaper and safer adaptation |
| QLoRA | Any of the above | LoRA adapters on quantized base | You want to fine-tune a larger model on limited GPU memory |
| Preference tuning | Chosen/rejected responses | Model preference behavior | You want outputs humans prefer, often after SFT |

## 2. The Core Objective

Most decoder-only LLM fine-tuning still uses next-token prediction. For supervised fine-tuning, we format examples so the model sees a prompt and desired answer. The loss is usually applied only to the assistant answer tokens, not to the user prompt tokens.

Conceptually:

```text
input_ids:  [system tokens, user tokens, assistant answer tokens]
labels:     [-100,         -100,        assistant answer token ids]
```

The value `-100` is ignored by PyTorch cross entropy. That is how we avoid training the model to predict the prompt itself.

In [ ]:
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
import json
import random
import statistics

import pandas as pd

random.seed(42)

## 3. A Tiny Instruction Dataset

Instruction data usually has at least an instruction and an answer. Many datasets also include context, system messages, metadata, difficulty, source, or quality labels.

For learning, start with tiny records you can inspect manually. Real fine-tuning fails quietly when the data format is wrong, so the habit of inspection matters.

In [ ]:
examples = [
    {
        "instruction": "Explain LoRA in simple terms.",
        "response": "LoRA freezes the original model and trains small low-rank matrices that adjust selected weight layers.",
        "domain": "finetuning",
    },
    {
        "instruction": "When should I use continued pretraining?",
        "response": "Use continued pretraining when the model needs more exposure to domain language, terminology, or style before instruction tuning.",
        "domain": "data",
    },
    {
        "instruction": "What does QLoRA save?",
        "response": "QLoRA saves GPU memory by loading the base model in 4-bit precision while training small LoRA adapter weights.",
        "domain": "memory",
    },
    {
        "instruction": "Name two common fine-tuning risks.",
        "response": "Two common risks are overfitting to a small dataset and forgetting useful behavior from the base model.",
        "domain": "evaluation",
    },
]

df = pd.DataFrame(examples)
df

## 4. Common Dataset Formats

You will see several formats in the wild. They can all represent similar information, but training code needs one consistent schema.

| Format | Example fields | Good for |
|---|---|---|
| Alpaca style | `instruction`, `input`, `output` | Simple instruction tuning |
| ShareGPT style | `conversations` list | Multi-turn chat |
| OpenAI chat style | `messages` list with roles | Chat models and APIs |
| Preference style | `prompt`, `chosen`, `rejected` | DPO, ORPO, reward modeling |
| Raw text | `text` | Continued pretraining |

In [ ]:
def to_openai_messages(example):
    return {
        "messages": [
            {"role": "system", "content": "You are a concise ML engineering tutor."},
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": example["response"]},
        ],
        "domain": example["domain"],
    }

chat_examples = [to_openai_messages(example) for example in examples]
print(json.dumps(chat_examples[0], indent=2))

## 5. Train, Validation, and Test Splits

A common mistake is to train and evaluate on near-duplicate examples. For fine-tuning, validation examples should test whether the behavior generalizes, not whether the model memorized the exact prompt.

Practical split guidance:

- keep related examples in the same split when possible,
- deduplicate before splitting,
- reserve examples that represent the hardest or most important use cases,
- keep a tiny hand-written regression set for behavior you never want to lose.

In [ ]:
def train_eval_split(records, eval_fraction=0.25, seed=42):
    records = list(records)
    rng = random.Random(seed)
    rng.shuffle(records)
    eval_size = max(1, round(len(records) * eval_fraction))
    return records[eval_size:], records[:eval_size]

train_records, eval_records = train_eval_split(chat_examples)
print(f"train: {len(train_records)}")
print(f"eval: {len(eval_records)}")
print(json.dumps(eval_records[0], indent=2))

## 6. Basic Data Quality Checks

Before training, check for missing fields, empty answers, duplicated prompts, very long examples, label leakage, and inconsistent style. These checks are simple, but they prevent many expensive fine-tuning mistakes.

In [ ]:
def extract_user_and_assistant(record):
    user_text = ""
    assistant_text = ""
    for message in record["messages"]:
        if message["role"] == "user":
            user_text += message["content"] + "\n"
        elif message["role"] == "assistant":
            assistant_text += message["content"] + "\n"
    return user_text.strip(), assistant_text.strip()

def validate_chat_records(records):
    issues = []
    seen_prompts = Counter()
    for index, record in enumerate(records):
        messages = record.get("messages", [])
        roles = [message.get("role") for message in messages]
        user_text, assistant_text = extract_user_and_assistant(record)
        seen_prompts[user_text] += 1
        if not messages:
            issues.append((index, "missing_messages"))
        if "user" not in roles:
            issues.append((index, "missing_user_message"))
        if "assistant" not in roles:
            issues.append((index, "missing_assistant_message"))
        if not assistant_text:
            issues.append((index, "empty_assistant_answer"))
        if len(assistant_text.split()) < 3:
            issues.append((index, "very_short_answer"))
    duplicate_prompts = [prompt for prompt, count in seen_prompts.items() if count > 1]
    return issues, duplicate_prompts

issues, duplicate_prompts = validate_chat_records(chat_examples)
print(f"issues: {issues}")
print(f"duplicate_prompts: {duplicate_prompts}")

## 7. Length and Cost Awareness

Long examples cost more memory and compute. The real unit is tokens, not characters or words. When a tokenizer is unavailable, word counts are only a rough proxy.

In [ ]:
def format_messages_as_text(messages):
    lines = []
    for message in messages:
        lines.append(f"<{message['role']}>\n{message['content']}")
    return "\n".join(lines)

def rough_token_count(text):
    return max(1, len(text.split()))

length_rows = []
for record in chat_examples:
    text = format_messages_as_text(record["messages"])
    user_text, assistant_text = extract_user_and_assistant(record)
    length_rows.append({
        "domain": record["domain"],
        "prompt_words": rough_token_count(user_text),
        "answer_words": rough_token_count(assistant_text),
        "total_words": rough_token_count(text),
    })

length_df = pd.DataFrame(length_rows)
length_df.describe()

## 8. Save JSONL Splits

Most fine-tuning pipelines accept JSONL: one JSON record per line. This makes large datasets streamable and easy to inspect with command-line tools.

In [ ]:
data_dir = Path("finetuning_data")
data_dir.mkdir(exist_ok=True)

def write_jsonl(path, records):
    with path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")

write_jsonl(data_dir / "train.jsonl", train_records)
write_jsonl(data_dir / "eval.jsonl", eval_records)

print((data_dir / "train.jsonl").read_text()[:500])

## 9. Data Quality Rubric

Use this quick rubric before fine-tuning:

| Question | Why it matters |
|---|---|
| Are answers correct? | Fine-tuning amplifies errors |
| Are examples diverse? | Narrow data causes brittle behavior |
| Are prompts realistic? | Synthetic prompts can teach unnatural behavior |
| Are answers consistent in style? | Inconsistent tone creates unstable outputs |
| Is eval data held out? | You need a real signal after training |
| Are long examples intentional? | Long context increases cost |
| Are refusals and boundaries represented? | Safety behavior can drift |
| Are duplicates removed? | Duplicates distort loss and eval |

## 10. When Not to Fine-Tune

Do not fine-tune just because prompts are imperfect. Consider simpler options first:

- prompt engineering when behavior is easy to describe,
- RAG when the model needs external facts,
- tool use when the model needs reliable actions or computation,
- routing when one model cannot handle every task,
- evaluation improvements when you cannot measure quality yet.

Fine-tuning is best when you have repeatable examples of behavior you want the model to learn.

## Summary

Fine-tuning is not one thing. It includes continued pretraining, supervised fine-tuning, instruction tuning, full fine-tuning, adapter methods, LoRA, QLoRA, and preference tuning. The common foundation is data: format, quality, length, split strategy, and labels.

Next, we will go deeper into chat templates and loss masking, because formatting decides exactly what the model is trained to predict.